<a href="https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1.Method Choice

I will use a **Random Forest Classifier** for the content opportunity lane.

The target is binary: whether a page is observed as declining or not declining. Random Forest fits this problem because it can capture non-linear relationships between search performance, content characteristics, freshness, and engagement signals without requiring a linear relationship. It also provides feature importance, which is useful for interpreting which observed signals the model relies on.

The model is used for **decision-support**, not as proof that a content refresh will cause improvement.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Setup: clone the repo in Colab and load the starter dataset

import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/usman-stack-322/flyrank-ml-internship-v2"
REPO_DIR = "flyrank-ml-internship-v2"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Current directory:", os.getcwd())
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Current directory: /content/flyrank-ml-internship-v2/flyrank-ml-internship-v2
Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [14]:
# Check the target used for modeling

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


In [15]:
# Confirm the modeling setup

target = "is_declining_label"

print("Task: Binary classification")
print("Target:", target)
print("Classes:", sorted(df[target].unique()))
print("Rows:", len(df))

Task: Binary classification
Target: is_declining_label
Classes: [np.int64(0), np.int64(1)]
Rows: 30000


## 2. Split design


I use a **time-aware split** for this task. The goal is to predict whether a page is declining using information that would have been available before the prediction moment.

A random split could place observations from the same time period on both sides and make the evaluation less realistic. A time-aware split is more honest for decision-support because the model is trained on earlier observations and evaluated on later observations.

The dataset contains historical 90-day and 30-day performance signals, so the split should preserve the intended prediction direction rather than allowing future information into training.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Time-aware split
# Use the row order as the available ordering because the dataset
# does not provide a separate observation-date column.

split_point = int(len(df) * 0.80)

train_df = df.iloc[:split_point].copy()
test_df = df.iloc[split_point:].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining target rate:",
      round(train_df["is_declining_label"].mean(), 3))

print("Test target rate:",
      round(test_df["is_declining_label"].mean(), 3))


Training rows: 24000
Test rows: 6000

Training target rate: 0.541
Test target rate: 0.547


## 3. Train + compare vs my baseline


I train a Random Forest classifier using the same time-aware split defined above. The model is compared with the Week-4 baseline on the same test rows.

I use precision, recall, and F1 as classification metrics. Precision measures how often predicted declining pages are actually declining, while recall measures how many observed declining pages are identified. F1 provides a balance between precision and recall.

The comparison is intended for decision-support. A better test metric does not prove that a content refresh will cause improvement.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Train Random Forest and compare it with the transparent baseline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

target = "is_declining_label"

# Do NOT use identifiers, label-source columns, or post-outcome fields.
feature_columns = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier",
]

X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = [
    col for col in feature_columns
    if col not in numeric_features
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# Train only on the earlier portion
pipeline.fit(X_train, y_train)

# Model predictions on the later portion
model_pred = pipeline.predict(X_test)

# Metrics
model_precision = precision_score(y_test, model_pred, zero_division=0)
model_recall = recall_score(y_test, model_pred, zero_division=0)
model_f1 = f1_score(y_test, model_pred, zero_division=0)

print("MODEL RESULTS")
print("Precision:", round(model_precision, 3))
print("Recall:   ", round(model_recall, 3))
print("F1:       ", round(model_f1, 3))


MODEL RESULTS
Precision: 0.811
Recall:    0.866
F1:        0.837


In [18]:
# Recreate the Week-4 baseline on the test set
# The baseline uses only information available to the rule.

test_baseline = test_df.copy()

# Position-tier CTR benchmarks calculated from TRAINING data only.
# This avoids using test-set information to construct the baseline.
benchmark = {
    tier: (
        train_df.loc[
            train_df["position_tier"] == tier,
            "clicks_90d"
        ].sum()
        /
        train_df.loc[
            train_df["position_tier"] == tier,
            "impressions_90d"
        ].sum()
        * 100
    )
    for tier in ["top_3", "page_1", "striking", "page_3_5", "deep"]
}

test_baseline["position_ctr_benchmark"] = (
    test_baseline["position_tier"].map(benchmark)
)

visible = (test_baseline["impressions_90d"] >= 300)

stale = (test_baseline["days_since_last_update"] >= 90)

ctr_gap = (
    test_baseline["position_tier"].isin(
        ["top_3", "page_1", "striking", "page_3_5"]
    )
    &
    (
        test_baseline["ctr"]
        < test_baseline["position_ctr_benchmark"]
    )
)

baseline_pred = (visible & stale & ctr_gap).astype(int)

baseline_precision = precision_score(
    y_test, baseline_pred, zero_division=0
)

baseline_recall = recall_score(
    y_test, baseline_pred, zero_division=0
)

baseline_f1 = f1_score(
    y_test, baseline_pred, zero_division=0
)

comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "precision": [baseline_precision, model_precision],
    "recall": [baseline_recall, model_recall],
    "f1": [baseline_f1, model_f1]
})

print(comparison.round(3).to_string(index=False))

         method  precision  recall    f1
Week-4 baseline      0.690   0.208 0.320
  Random Forest      0.811   0.866 0.837


### Model vs Baseline

On the same time-aware test split, the Random Forest achieved a precision of 0.811, recall of 0.866, and F1 of 0.837. The Week-4 baseline achieved 0.690 precision, 0.208 recall, and 0.320 F1.

The model therefore showed a substantially stronger measured balance of precision and recall than the transparent baseline on this test split. This is evidence that the model is better at identifying observed declining pages in this evaluation, but it does not prove that a content refresh will cause those pages to improve.


## 4. Errors and Interpretation

The Random Forest's strongest observed signals are recent and previous-period impressions, followed by 90-day impressions, days with impressions, average position, and content age. This suggests the model is relying heavily on recent search visibility and page performance rather than a single rule such as staleness or CTR alone.

The model still produces false positives and false negatives, so its predictions are not perfect. These errors can occur when observed search signals do not fully capture why a page is declining. The model should therefore be used as decision-support, with human review before taking a refresh action.



In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Error analysis on the time-aware test set

error_analysis = test_df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "position_tier",
    "content_age_days",
    "trend_direction",
    "is_declining_label"
]].copy()

error_analysis["prediction"] = model_pred

# False positives: predicted declining, actually not declining
false_positives = error_analysis[
    (error_analysis["prediction"] == 1) &
    (error_analysis["is_declining_label"] == 0)
]

# False negatives: predicted not declining, actually declining
false_negatives = error_analysis[
    (error_analysis["prediction"] == 0) &
    (error_analysis["is_declining_label"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse-positive rate among predictions:")
print(round(len(false_positives) / len(error_analysis), 3))

print("\nFalse-negative rate among predictions:")
print(round(len(false_negatives) / len(error_analysis), 3))

print("\nFalse-positive examples:")
display(false_positives.head(5))

print("\nFalse-negative examples:")
display(false_negatives.head(5))

False positives: 663
False negatives: 441

False-positive rate among predictions:
0.111

False-negative rate among predictions:
0.073

False-positive examples:


,content_id,client_id,impressions_90d,days_since_last_update,ctr,avg_position,position_tier,content_age_days,trend_direction,is_declining_label,prediction
24005,content_4ae53d409dbe,client_6208ef0f77,1210,20,0.41,7.8,page_1,118,up,0,1
24026,content_9a090e5c6a44,client_3fdba35f04,530,104,0.38,7.5,page_1,131,stable,0,1
24040,content_1063b1a90217,client_7f2253d7e2,4648,20,0.28,20.6,page_3_5,230,stable,0,1
24065,content_07211ca13e10,client_4e07408562,2548,13,0.24,11.0,striking,228,up,0,1
24077,content_dff3a7e0db49,client_6208ef0f77,1674,104,0.18,12.4,striking,287,stable,0,1



False-negative examples:


,content_id,client_id,impressions_90d,days_since_last_update,ctr,avg_position,position_tier,content_age_days,trend_direction,is_declining_label,prediction
24018,content_1caa959588a1,client_6208ef0f77,44976,104,0.10,30.3,page_3_5,211,down,1,0
24038,content_469526eec435,client_e629fa6598,6282,22,0.64,3.8,page_1,460,down,1,0
24050,content_7ea36c800544,client_4e07408562,13630,25,0.16,9.0,page_1,421,down,1,0
24061,content_d73157d3477f,client_19581e27de,10171,20,0.15,8.6,page_1,463,down,1,0
24084,content_579c1f2bf237,client_19581e27de,2431,22,0.04,12.9,striking,482,down,1,0


In [20]:
# Inspect which transformed features the Random Forest relies on most

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("Top 15 model features:")
display(importance_df.head(15))

Top 15 model features:


,feature,importance
18,numeric__impressions_prev_30d,0.262804
15,numeric__impressions_last_30d,0.145609
5,numeric__impressions_90d,0.072515
13,numeric__days_with_impressions,0.066847
25,numeric__avg_position,0.051029
21,numeric__content_age_days,0.043301
16,numeric__clicks_last_30d,0.031421
17,numeric__sessions_last_30d,0.023260
4,numeric__char_count,0.017922
22,numeric__age_tier_order,0.017691


In [21]:
# Count model errors on the test set

false_positives = (
    (model_pred == 1) & (y_test.to_numpy() == 0)
).sum()

false_negatives = (
    (model_pred == 0) & (y_test.to_numpy() == 1)
).sum()

print("False positives:", false_positives)
print("False negatives:", false_negatives)

False positives: 663
False negatives: 441


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.